# LifeMate Agent — Part B: ASR & Audio Normalization (Colab / Kaggle)
**Team 3 — Member 2 Specialist Guide**

This notebook benchmarks and runs the Vietnamese ASR pipeline with:
- **faster-whisper** (`large-v3-turbo`, `medium`, `small`)
- **Contextual Prompt Biasing** (injecting app names & tool commands into initial prompt)
- **Unicode NFC Normalization & Number/Date Normalizer**
- **WER & CER Evaluation** using `jiwer` against ground truth transcripts.

## 1. Environment Setup (Run once on Colab / Kaggle T4)

In [ ]:
!nvidia-smi
!pip install -q faster-whisper librosa soundfile jiwer tqdm pandas

## 2. Mount Data & Configure Paths

In [ ]:
import os, sys, json
from pathlib import Path

# If using Google Drive in Colab:
# from google.colab import drive
# drive.mount('/content/drive')

DATA_DIR = Path("./train+public_test")
if not DATA_DIR.exists():
    print("⚠️ Please make sure 'train+public_test' folder is in current directory!")
else:
    print("✅ Data directory found:", list(DATA_DIR.iterdir()))

## 3. Vietnamese Audio Normalizer

In [ ]:
import unicodedata, re

def normalize_vietnamese_text(text: str) -> str:
    if not text:
        return ""
    # Standardize to Unicode NFC
    text = unicodedata.normalize("NFC", text)
    # Strip redundant whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Test normalizer
sample_str = "nạp   50 nghìn   điện thoại"
print("Normalized:", normalize_vietnamese_text(sample_str))

## 4. Load faster-whisper on GPU T4

In [ ]:
from faster_whisper import WhisperModel
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
compute_type = "float16" if device == "cuda" else "int8"

print(f"Loading faster-whisper large-v3-turbo on {device} ({compute_type})...")
model = WhisperModel("large-v3-turbo", device=device, compute_type=compute_type)
print("✅ Model loaded successfully!")

## 5. Benchmark ASR on Dev Set (WER / CER / Speed)

In [ ]:
import time, jiwer
from tqdm import tqdm

# Load dev set IDs
with open(DATA_DIR / "train" / "train.jsonl", encoding="utf-8") as f:
    all_train = [json.loads(line) for line in f if line.strip()]

print(f"Loaded {len(all_train)} training samples. Running benchmark on first 50 audio samples...")
bench_samples = all_train[:50]

predictions = []
references = []
times = []

# Initial prompt with Vietnamese domain terms & app names
INITIAL_PROMPT = "Trợ lý ảo LifeMate, điều khiển nhà thông minh, chuyển khoản ngân hàng, đặt xe, nạp tiền điện thoại, nhắc nhở, lịch hẹn."

for s in tqdm(bench_samples):
    audio_path = DATA_DIR / "train" / "audio" / f"{s['id']}.wav"
    if not audio_path.exists():
        continue
    
    t0 = time.time()
    segments, _ = model.transcribe(
        str(audio_path),
        language="vi",
        initial_prompt=INITIAL_PROMPT,
        beam_size=3
    )
    pred_text = normalize_vietnamese_text(" ".join([seg.text for seg in segments]))
    elapsed = time.time() - t0
    
    gold_text = normalize_vietnamese_text(s.get("audio_text", ""))
    predictions.append(pred_text)
    references.append(gold_text)
    times.append(elapsed)

wer = jiwer.wer(references, predictions)
cer = jiwer.cer(references, predictions)
avg_time = sum(times) / len(times) if times else 0

print("=" * 60)
print(f"ASR BENCHMARK RESULTS (large-v3-turbo)")
print(f"WER: {wer * 100:.2f}%")
print(f"CER: {cer * 100:.2f}%")
print(f"Avg Inference Time: {avg_time:.2f}s / sample (Target: < 5s)")
print("=" * 60)

## 6. Inspect Sample Transcriptions

In [ ]:
for i in range(min(5, len(predictions))):
    print(f"[{bench_samples[i]['id']}]")
    print(f"  Ref : {references[i]}")
    print(f"  Pred: {predictions[i]}")
    print()
